# RL-Swing-Trader v6 (clean)
위에서부터 순서대로 실행. 셀 7은 VAL 결과로 최종 seed를 고른 뒤 **딱 한 번만** 실행.

In [32]:
# [Cell 1] Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
# [Cell 2] 설치 & import (세션 새로 시작할 때마다 필요)
!pip install yfinance gymnasium "stable-baselines3[extra]" --quiet

import os, gc
import numpy as np
import pandas as pd
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback

print("✅ 준비 완료 | GPU:", torch.cuda.is_available())

✅ 준비 완료 | GPU: False


In [34]:
# [Cell 3] 설정 & 데이터
TRAIN_TICKERS = ["AAPL", "SOFI", "F", "PEP", "MSFT", "JNJ", "XOM", "T", "DIS",
                 "NKE", "WMT", "BA", "GE", "INTC"]
HOLDOUT_TICKERS = ["KO"]   # 훈련/튜닝에 절대 안 쓰는 종목

START_DATE = "2000-01-01"
END_DATE   = "2026-09-30"  # 2026년 데이터까지 → 모델이 한 번도 못 본 '진짜 미래' 구간

# 날짜 기준 분할: 모든 종목이 같은 달력을 공유
TRAIN_END = "2022-12-31"   # TRAIN : 학습
VAL_END   = "2024-12-31"   # VAL   : seed / 리워드 비교는 여기서만
TEST_END  = "2025-12-31"   # TEST  : 최종 모델 확정 후 한 번만
                           # FORWARD (2026-01-01~): 가장 정직한 마지막 시험

def download(ticker):
    df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()

data = {t: download(t) for t in TRAIN_TICKERS + HOLDOUT_TICKERS}
for t, df in data.items():
    print(f"{t}: {len(df)}일 ({df.index[0].date()} ~ {df.index[-1].date()})")

AAPL: 6725일 (2000-01-03 ~ 2026-09-29)
SOFI: 1441일 (2021-01-04 ~ 2026-09-29)
F: 6725일 (2000-01-03 ~ 2026-09-29)
PEP: 6725일 (2000-01-03 ~ 2026-09-29)
MSFT: 6725일 (2000-01-03 ~ 2026-09-29)
JNJ: 6725일 (2000-01-03 ~ 2026-09-29)
XOM: 6725일 (2000-01-03 ~ 2026-09-29)
T: 6725일 (2000-01-03 ~ 2026-09-29)
DIS: 6725일 (2000-01-03 ~ 2026-09-29)
NKE: 6725일 (2000-01-03 ~ 2026-09-29)
WMT: 6725일 (2000-01-03 ~ 2026-09-29)
BA: 6725일 (2000-01-03 ~ 2026-09-29)
GE: 6725일 (2000-01-03 ~ 2026-09-29)
INTC: 6725일 (2000-01-03 ~ 2026-09-29)
KO: 6725일 (2000-01-03 ~ 2026-09-29)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [35]:
# [Cell 4] 피처 계산 (전체 시계열에서 한 번만 → 워밍업 왜곡 없음 + 훈련 속도 향상)
def add_features(df):
    close = df["Close"]
    out = pd.DataFrame(index=df.index)
    out["close"]    = close
    out["mom5"]     = close.pct_change(5)                      # 단기 모멘텀
    out["ma20_dev"] = close / close.rolling(20).mean() - 1     # 20일선 대비 위치
    out["ma60_dev"] = close / close.rolling(60).mean() - 1     # 60일선 대비 위치 (긴 추세)
    out["dd60"]     = close / close.rolling(60).max() - 1      # 60일 고점 대비 낙폭 (떨어지는 칼날 감지)
    out["vol20"]    = close.pct_change().rolling(20).std()     # 최근 변동성
    return out.dropna()   # 앞쪽 60일 워밍업 제거

feat = {t: add_features(df) for t, df in data.items()}

splits = {
    "train":   {t: feat[t].loc[:TRAIN_END] for t in TRAIN_TICKERS},
    "val":     {t: feat[t].loc["2023-01-01":VAL_END] for t in TRAIN_TICKERS},
    "test":    {t: feat[t].loc["2025-01-01":TEST_END] for t in TRAIN_TICKERS},
    "forward": {t: feat[t].loc["2026-01-01":] for t in TRAIN_TICKERS},
    "holdout": {t: feat[t].loc["2023-01-01":] for t in HOLDOUT_TICKERS},
}
for name, d in splits.items():
    print(f"{name:8s}: " + ", ".join(f"{t}={len(df)}" for t, df in d.items()))

train   : AAPL=5728, SOFI=444, F=5728, PEP=5728, MSFT=5728, JNJ=5728, XOM=5728, T=5728, DIS=5728, NKE=5728, WMT=5728, BA=5728, GE=5728, INTC=5728
val     : AAPL=502, SOFI=502, F=502, PEP=502, MSFT=502, JNJ=502, XOM=502, T=502, DIS=502, NKE=502, WMT=502, BA=502, GE=502, INTC=502
test    : AAPL=250, SOFI=250, F=250, PEP=250, MSFT=250, JNJ=250, XOM=250, T=250, DIS=250, NKE=250, WMT=250, BA=250, GE=250, INTC=250
forward : AAPL=186, SOFI=186, F=186, PEP=186, MSFT=186, JNJ=186, XOM=186, T=186, DIS=186, NKE=186, WMT=186, BA=186, GE=186, INTC=186
holdout : KO=938


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [36]:
# [Cell 5] 트레이딩 환경 v6
class SwingTradingEnv(gym.Env):
    FEATURES = ["mom5", "ma20_dev", "ma60_dev", "dd60", "vol20"]

    def __init__(self, data_dict, initial_balance=100.0, fee_percent=0.001,
                 min_holding_days=3, settle_days=1, episode_length=250, random_start=True,
                 loss_penalty_coef=5.0, streak_penalty_coef=0.3, blocked_sell_penalty=0.05):
        super().__init__()
        self.series = {t: (df["close"].values.astype(np.float32),
                           df[self.FEATURES].values.astype(np.float32))
                       for t, df in data_dict.items() if len(df) > 30}
        self.tickers = list(self.series.keys())
        self.initial_balance = initial_balance
        self.fee_percent = fee_percent            # ⚠️ 실제 Webull 수수료 + 환전비용으로 바꿀 것
        self.min_holding_days = min_holding_days
        self.settle_days = settle_days            # 미국/캐나다 모두 2024년부터 T+1
        self.episode_length = episode_length
        self.random_start = random_start
        self.loss_penalty_coef = loss_penalty_coef
        self.streak_penalty_coef = streak_penalty_coef
        self.blocked_sell_penalty = blocked_sell_penalty

        self.action_space = spaces.Discrete(3)    # 0=관망, 1=매수, 2=매도
        n_obs = 5 + len(self.FEATURES)            # 계좌 상태 5 + 시장 지표 5
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(n_obs,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)                  # seed 재현성: gym 내장 난수 사용
        self.ticker = self.tickers[self.np_random.integers(len(self.tickers))]
        prices, feats = self.series[self.ticker]
        ep_len = min(self.episode_length, len(prices) - 1)
        start = int(self.np_random.integers(0, len(prices) - ep_len)) if self.random_start else 0

        self.prices = prices[start:start + ep_len + 1]
        self.feats = feats[start:start + ep_len + 1]
        self.t = 0
        self.balance = self.initial_balance
        self.pending_cash = []
        self.shares = 0.0
        self.entry_step = None
        self.entry_price = None
        self.net_worth = self.initial_balance
        self.consecutive_losses = 0
        self.trade_count = 0
        self.benchmark_shares = self.initial_balance / self.prices[0]
        self.benchmark_worth = self.initial_balance
        return self._get_obs(), {}

    def _settle(self):
        still = []
        for item in self.pending_cash:
            if item["settle_step"] <= self.t:
                self.balance += item["amount"]
            else:
                still.append(item)
        self.pending_cash = still

    def _get_obs(self):
        i = min(self.t, len(self.prices) - 1)
        price = self.prices[i]
        pending = sum(p["amount"] for p in self.pending_cash)
        worth = max(self.net_worth, 1e-6)
        in_pos = 1.0 if self.shares > 0 else 0.0
        unrealized = (price / self.entry_price - 1) if self.shares > 0 else 0.0   # 지금 이 거래가 수익/손실 중인지
        hold_prog = 0.0
        if self.entry_step is not None:
            hold_prog = min((self.t - self.entry_step) / self.min_holding_days, 1.0)
        account = np.array([self.balance / worth, pending / worth, in_pos, unrealized, hold_prog],
                           dtype=np.float32)
        return np.concatenate([account, self.feats[i]]).astype(np.float32)

    def step(self, action):
        action = int(action)
        price = self.prices[self.t]
        self._settle()
        extra = 0.0

        if action == 1 and self.shares == 0 and self.balance > 0:
            fee = self.balance * self.fee_percent
            self.shares = (self.balance - fee) / price
            self.balance = 0.0
            self.entry_step, self.entry_price = self.t, price
            self.trade_count += 1

        elif action == 2 and self.shares > 0:
            if self.t - self.entry_step < self.min_holding_days:
                extra -= self.blocked_sell_penalty
            else:
                revenue = self.shares * price
                self.pending_cash.append({"amount": revenue * (1 - self.fee_percent),
                                          "settle_step": self.t + self.settle_days})
                if price < self.entry_price:
                    self.consecutive_losses += 1
                    # 연속 손실 페널티: 손실 매도 '그 순간'에 한 번만 (매 스텝 X)
                    extra -= self.streak_penalty_coef * self.consecutive_losses
                else:
                    self.consecutive_losses = 0
                self.shares = 0.0
                self.entry_step = self.entry_price = None
                self.trade_count += 1

        self.t += 1
        i = min(self.t, len(self.prices) - 1)
        next_price = self.prices[i]
        pending = sum(p["amount"] for p in self.pending_cash)
        new_worth = self.balance + pending + self.shares * next_price

        pct = new_worth / max(self.net_worth, 1e-6) - 1
        new_bench = self.benchmark_shares * next_price
        bench_pct = new_bench / max(self.benchmark_worth, 1e-6) - 1
        self.benchmark_worth = new_bench
        self.net_worth = new_worth

        reward = (pct - bench_pct) * 100 + min(pct, 0) * self.loss_penalty_coef + extra
        terminated = self.t >= len(self.prices) - 1
        info = {"net_worth": new_worth, "trades": self.trade_count}
        return self._get_obs(), float(reward), terminated, False, info


def evaluate(model, data_dict):
    rows = []
    for t, df in data_dict.items():
        if len(df) <= 30:
            continue
        env = SwingTradingEnv({t: df}, episode_length=len(df) - 1, random_start=False)
        obs, _ = env.reset()
        done, worths, info = False, [env.initial_balance], {"trades": 0}
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, _, info = env.step(action)
            worths.append(info["net_worth"])
        worths = np.array(worths)
        max_dd = (worths / np.maximum.accumulate(worths) - 1).min() * 100
        bh = 100 * df["close"].iloc[-1] / df["close"].iloc[0]
        rows.append({"ticker": t, "bot": round(float(worths[-1]), 2), "buy_hold": round(float(bh), 2),
                     "beat": bool(worths[-1] > bh), "trades": info["trades"],
                     "max_dd_%": round(float(max_dd), 1)})
    return pd.DataFrame(rows)

print("✅ SwingTradingEnv v6 준비 완료")

✅ SwingTradingEnv v6 준비 완료


In [37]:
# [Cell 6] 빠른 스모크 테스트 (본 훈련 전, 에러 확인용 ~몇 분)
quick = PPO("MlpPolicy", DummyVecEnv([lambda: Monitor(SwingTradingEnv(splits["train"]))]),
            seed=0, n_steps=2048, verbose=0)
quick.learn(total_timesteps=10_000)
print(evaluate(quick, splits["val"]).head())
del quick; gc.collect()

  ticker    bot  buy_hold   beat  trades  max_dd_%
0   AAPL  100.0    202.33  False       0       0.0
1   SOFI  100.0    342.22  False       0       0.0
2      F  100.0    100.22  False       0       0.0
3    PEP  100.0     89.98   True       0       0.0
4   MSFT  100.0    178.82  False       0       0.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


464

In [38]:
# [Cell 7] 본 훈련 + VAL 평가 (seed 비교는 VAL로만!)
SAVE_DIR = "/content/drive/MyDrive/RL-Swing-Trader/v8_since2000"
LOSS_COEF = 1.0
SEEDS = [0, 42, 100]
TIMESTEPS = 500_000
os.makedirs(SAVE_DIR, exist_ok=True)

def make_env():
    return Monitor(SwingTradingEnv(splits["train"], episode_length=250, random_start=True,
                                   loss_penalty_coef=LOSS_COEF))
all_val = []

for seed in SEEDS:
    path = f"{SAVE_DIR}/seed{seed}_final.zip"
    if os.path.exists(path):
        print(f"seed={seed}: 저장된 모델 불러옴")
        model = PPO.load(path)
    else:
        print(f"seed={seed}: 훈련 시작")
        env = DummyVecEnv([make_env])
        ckpt = CheckpointCallback(save_freq=50_000, save_path=f"{SAVE_DIR}/ckpt_seed{seed}/",
                                  name_prefix="swing")
        model = PPO("MlpPolicy", env, seed=seed, ent_coef=0.001, n_steps=4096, verbose=0)
        model.learn(total_timesteps=TIMESTEPS, callback=ckpt)
        model.save(path)
        del env

    res = evaluate(model, splits["val"])
    res["seed"] = seed
    all_val.append(res)
    print(f"  VAL 평균 ${res.bot.mean():.2f} | B&H 승률 {res.beat.mean():.0%} | "
          f"평균 거래 {res.trades.mean():.1f}회 | 평균 MDD {res['max_dd_%'].mean():.1f}%")
    del model; gc.collect()

val_df = pd.concat(all_val)
val_df.to_csv(f"{SAVE_DIR}/val_results.csv", index=False)
print(val_df.pivot(index="ticker", columns="seed", values="bot"))

seed=0: 훈련 시작


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  VAL 평균 $138.76 | B&H 승률 7% | 평균 거래 1.9회 | 평균 MDD -29.5%
seed=42: 훈련 시작


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  VAL 평균 $135.51 | B&H 승률 7% | 평균 거래 7.4회 | 평균 MDD -29.5%
seed=100: 훈련 시작


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  VAL 평균 $147.60 | B&H 승률 29% | 평균 거래 4.4회 | 평균 MDD -28.4%
seed       0       42      100
ticker                        
AAPL    202.13  202.13  202.13
BA       90.50   80.10   91.56
DIS     104.67  101.64  130.53
F        94.23   91.95  100.12
GE      274.25  310.55  301.11
INTC     61.51   74.86   79.53
JNJ      86.18   86.18   86.18
MSFT    178.64  178.64  178.64
NKE      66.11   59.12   66.19
PEP      89.89   89.89   89.89
SOFI    255.34  174.76  301.33
T       137.56  145.73  137.56
WMT     193.74  193.74  193.74
XOM     107.90  107.90  107.90


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
# [Cell 8] ⚠️ 최종 시험: VAL로 BEST_SEED를 먼저 정한 뒤 딱 한 번만 실행
from stable_baselines3 import PPO

BEST_SEED = 100   # ← VAL 결과 보고 직접 선택 (TEST 결과 보고 바꾸면 반칙)
model = PPO.load(f"{SAVE_DIR}/seed{BEST_SEED}_final.zip")

for name in ["test", "holdout", "forward"]:
    df = evaluate(model, splits[name])
    print(f"\n===== {name.upper()} =====")
    print(df.to_string(index=False))
    print(f"B&H 승률 {df.beat.mean():.0%} | 평균 ${df.bot.mean():.2f} | 평균 MDD {df['max_dd_%'].mean():.1f}%")


===== TEST =====
ticker    bot  buy_hold  beat  trades  max_dd_%
  AAPL 111.88    111.99 False       1     -30.2
  SOFI 193.43    185.28  True      19     -43.6
     F 145.89    146.03 False       1     -15.6
   PEP  99.27     99.36 False       1     -17.8
  MSFT 116.27    116.39 False       1     -20.6
   JNJ 147.95    148.09 False       1     -12.7
   XOM 116.14    116.26 False       1     -16.1
     T 113.56    113.68 False       1     -17.9
   DIS 101.89    103.80 False       3     -28.3
   NKE  86.00     88.51 False       3     -34.5
   WMT 124.85    124.97 False       1     -21.9
    BA 126.20    126.33 False       1     -26.7
    GE 183.57    183.75 False       1     -21.4
  INTC 161.54    182.49 False      15     -25.3
B&H 승률 7% | 평균 $130.60 | 평균 MDD -23.8%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



===== HOLDOUT =====
ticker    bot  buy_hold  beat  trades  max_dd_%
    KO 153.75    153.91 False       1     -17.3
B&H 승률 0% | 평균 $153.75 | 평균 MDD -17.3%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


===== FORWARD =====
ticker    bot  buy_hold  beat  trades  max_dd_%
  AAPL 121.75    121.88 False       1     -12.7
  SOFI  59.61     57.94  True       3     -48.3
     F 100.25     95.40  True       5     -22.3
   PEP  93.14     93.23 False       1     -22.5
  MSFT  96.68    108.30 False       3     -26.7
   JNJ 130.99    131.12 False       1     -11.0
   XOM 134.06    134.19 False       1     -20.1
     T 103.16    103.26 False       1     -28.9
   DIS  94.87     94.96 False       1     -20.2
   NKE  58.08     58.13 False       1     -45.0
   WMT  95.22     95.32 False       1     -23.3
    BA  82.32     82.40 False       1     -26.9
    GE  99.28     99.38 False       1     -20.9
  INTC 169.11    294.39 False      13     -41.4
B&H 승률 14% | 평균 $102.75 | 평균 MDD -26.4%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [40]:
# [Cell 9] 진단: loss penalty 크기에 따라 cash/hold/random 비교
def run_policy(policy, n_episodes=300, seed=0, **env_kwargs):
    env = SwingTradingEnv(splits["train"], episode_length=250, random_start=True, **env_kwargs)
    rng = np.random.default_rng(seed)
    obs, _ = env.reset(seed=seed)
    totals = []
    for _ in range(n_episodes):
        done, total = False, 0.0
        while not done:
            if policy == "cash":
                action = 0
            elif policy == "hold":
                action = 1
            else:
                action = rng.integers(3)
            obs, r, done, _, _ = env.step(action)
            total += r
        totals.append(total)
        obs, _ = env.reset()
    return np.mean(totals), np.std(totals)

for coef in [5.0, 1.0, 0.0]:
    print(f"\n--- loss_penalty_coef = {coef} ---")
    for p in ["cash", "hold", "random"]:
        m, s = run_policy(p, loss_penalty_coef=coef)
        print(f"{p:7s}: 평균 {m:8.2f}  (std {s:.2f})")


--- loss_penalty_coef = 5.0 ---


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


cash   : 평균    -6.15  (std 35.26)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


hold   : 평균    -9.17  (std 6.24)


...

cash   : 평균    -6.15  (std 35.26)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


hold   : 평균    -0.10  (std 0.00)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


random : 평균   -17.70  (std 21.56)


In [41]:
# [Cell 10] A. 공정 비교: v5 vs v6 (둘 다 처음 보는 2026 FORWARD 구간)
class SwingTradingEnvV5(gym.Env):
    def __init__(self, data_dict, initial_balance=100.0, fee_percent=0.001, min_holding_days=3,
                 episode_length=250, random_start=True):
        super().__init__()
        self.ticker_data = {t: np.asarray(df['Close']).flatten().astype(np.float32)
                            for t, df in data_dict.items()}
        self.tickers = list(self.ticker_data.keys())
        self.initial_balance = initial_balance
        self.fee_percent = fee_percent
        self.min_holding_days = min_holding_days
        self.episode_length = episode_length
        self.random_start = random_start
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(7,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        ticker = np.random.choice(self.tickers)
        full = self.ticker_data[ticker]
        ep_len = min(self.episode_length, len(full) - 1)
        start_idx = np.random.randint(0, max(len(full) - ep_len - 1, 1)) if self.random_start else 0
        self.data = full[start_idx: start_idx + ep_len + 1]
        self.current_step = 0
        self.balance = self.initial_balance
        self.pending_cash = []
        self.shares_held = 0.0
        self.entry_step = None
        self.net_worth = self.initial_balance
        self.consecutive_losses = 0
        self.benchmark_shares = self.initial_balance / self.data[0]
        self.benchmark_worth = self.initial_balance
        return self._get_obs(), {}

    def _settle_pending_cash(self):
        still = []
        for item in self.pending_cash:
            if item["settle_step"] <= self.current_step:
                self.balance += item["amount"]
            else:
                still.append(item)
        self.pending_cash = still

    def _get_obs(self):
        idx = min(self.current_step, len(self.data) - 1)
        price = self.data[idx]
        position_value = self.shares_held * price
        pending_value = sum(item["amount"] for item in self.pending_cash)
        holding_progress = 0.0
        if self.entry_step is not None:
            holding_progress = min((self.current_step - self.entry_step) / self.min_holding_days, 1.0)
        normalized_price = price / self.data[0]
        past_price = self.data[max(idx - 5, 0)]
        momentum = (price - past_price) / past_price if past_price != 0 else 0.0
        ma20 = np.mean(self.data[max(0, idx - 19):idx + 1])
        ma_deviation = (price - ma20) / ma20 if ma20 != 0 else 0.0
        return np.array([self.balance / self.initial_balance, position_value / self.initial_balance,
                         pending_value / self.initial_balance, holding_progress,
                         normalized_price, momentum, ma_deviation], dtype=np.float32)

    def step(self, action):
        current_price = self.data[self.current_step]
        self._settle_pending_cash()
        if action == 1 and self.balance > 0 and self.shares_held == 0:
            fee = self.balance * self.fee_percent
            self.shares_held = (self.balance - fee) / current_price
            self.balance = 0
            self.entry_step = self.current_step
            self.entry_price = current_price
        elif action == 2 and self.shares_held > 0:
            if self.current_step - self.entry_step >= self.min_holding_days:
                revenue = self.shares_held * current_price
                self.pending_cash.append({"amount": revenue - revenue * self.fee_percent,
                                          "settle_step": self.current_step + 2})
                self.shares_held = 0
                self.entry_step = None
        self.current_step += 1
        idx = min(self.current_step, len(self.data) - 1)
        pending_value = sum(item["amount"] for item in self.pending_cash)
        self.net_worth = self.balance + pending_value + self.shares_held * self.data[idx]
        terminated = self.current_step >= len(self.data) - 1
        return self._get_obs(), 0.0, terminated, False, {}   # 평가용이라 reward는 안 씀


def evaluate_v5(model, raw_dict):
    rows = []
    for t, df in raw_dict.items():
        env = SwingTradingEnvV5({t: df}, episode_length=len(df) - 1, random_start=False)
        obs, _ = env.reset()
        done, worths, trades = False, [env.initial_balance], 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            prev = env.shares_held
            obs, _, done, _, _ = env.step(int(action))
            if env.shares_held != prev:
                trades += 1
            worths.append(env.net_worth)
        worths = np.array(worths)
        mdd = (worths / np.maximum.accumulate(worths) - 1).min() * 100
        bh = 100 * df["Close"].iloc[-1] / df["Close"].iloc[0]
        rows.append({"ticker": t, "bot": round(float(worths[-1]), 2), "buy_hold": round(float(bh), 2),
                     "beat": bool(worths[-1] > bh), "trades": trades, "max_dd_%": round(float(mdd), 1)})
    return pd.DataFrame(rows)


# ===== 비교 실행 =====
V5_DIR = "/content/drive/MyDrive/RL-Swing-Trader/v5_trend_and_streak"
V6_DIR = "/content/drive/MyDrive/RL-Swing-Trader/v7_loss1"   # ← 셀 7에서 쓴 SAVE_DIR과 같은지 확인!

FWD = TRAIN_TICKERS + ["KO"]
raw_fwd  = {t: data[t].loc["2026-01-01":] for t in FWD}   # v5용: 원본 가격
feat_fwd = {t: feat[t].loc["2026-01-01":] for t in FWD}   # v6용: 피처 포함

summary = []
for seed in [0, 42, 100, 7, 2024]:
    for version, path, fn, d in [
        ("v5", f"{V5_DIR}/checkpoints_seed{seed}/swing_trader_300000_steps.zip", evaluate_v5, raw_fwd),
        ("v6", f"{V6_DIR}/seed{seed}_final.zip", evaluate, feat_fwd),
    ]:
        if not os.path.exists(path):
            print(f"⚠️ 파일 없음: {path}")
            continue
        model = PPO.load(path)
        res = fn(model, d)
        summary.append({"version": version, "seed": seed,
                        "avg_$": round(res.bot.mean(), 2),
                        "beat_%": round(res.beat.mean() * 100),
                        "avg_trades": round(res.trades.mean(), 1),
                        "avg_mdd_%": round(res["max_dd_%"].mean(), 1)})
        del model; gc.collect()

bh_avg = np.mean([100 * df["Close"].iloc[-1] / df["Close"].iloc[0] for df in raw_fwd.values()])
print(pd.DataFrame(summary).sort_values(["version", "seed"]).to_string(index=False))
print(f"\n📊 Buy&Hold 평균 (2026 FORWARD, 15종목): ${bh_avg:.2f}")

version  seed  avg_$  beat_%  avg_trades  avg_mdd_%
     v5     0 112.46      60         1.2      -23.6
     v5     7 100.00      53         0.0        0.0
     v5    42 115.05       7         1.3      -25.7
     v5   100  99.71      53         0.3       -4.7
     v5  2024 106.73       7         1.6      -24.3
     v6     0 102.01      20         2.7      -24.9
     v6     7 111.65       0         1.3      -24.5
     v6    42 113.09       0         1.0      -25.7
     v6   100 106.98       7         1.5      -25.4
     v6  2024 113.09       0         1.0      -25.7

📊 Buy&Hold 평균 (2026 FORWARD, 15종목): $113.20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [42]:
# [Cell 11] 진단: 60일선 추세 규칙이 Buy&Hold보다 나은가?
class TrendRule:
    """AI 대신 사람이 만든 규칙: 60일선 위면 보유(1), 아래면 매도(2)"""
    def predict(self, obs, deterministic=True):
        ma60_dev = obs[7]          # 관측값 순서: 계좌 5개 다음 [mom5, ma20_dev, ma60_dev, dd60, vol20]
        return (1 if ma60_dev > 0 else 2), None

def bh_mdd(df):
    """Buy&Hold의 최대 낙폭 (비교용)"""
    w = df["close"].values / df["close"].values[0]
    return (w / np.maximum.accumulate(w) - 1).min() * 100

for name, d in [("VAL", splits["val"]), ("FORWARD", feat_fwd)]:
    res = evaluate(TrendRule(), d)
    res["bh_mdd_%"] = [round(bh_mdd(d[t]), 1) for t in res.ticker]
    print(f"\n===== {name} =====")
    print(res.to_string(index=False))
    print(f"규칙:     평균 ${res.bot.mean():.2f} | 평균 MDD {res['max_dd_%'].mean():.1f}%")
    print(f"Buy&Hold: 평균 ${res.buy_hold.mean():.2f} | 평균 MDD {res['bh_mdd_%'].mean():.1f}%")


===== VAL =====
ticker    bot  buy_hold  beat  trades  max_dd_%  bh_mdd_%
  AAPL 149.20    202.33 False      25     -16.1     -16.6
  SOFI 124.40    342.22 False      25     -65.1     -44.8
     F  79.58    100.22 False      26     -31.9     -35.7
   PEP  82.59     89.98 False      42     -25.4     -18.7
  MSFT 136.31    178.82 False      30     -17.4     -15.5
   JNJ  87.40     86.27  True      26     -19.6     -17.4
   XOM  84.94    108.01 False      40     -22.1     -18.7
     T 136.49    137.70 False      15     -18.9     -32.0
   DIS 119.69    126.70 False      17     -32.1     -30.0
   NKE  66.83     65.65  True      24     -39.3     -44.0
   WMT 161.74    193.93 False      23     -13.3     -10.8
    BA  88.04     90.59 False      33     -36.3     -47.7
    GE 233.43    318.07 False      26     -19.1     -17.6
  INTC  96.87     77.79  True      30     -34.6     -62.3
규칙:     평균 $117.68 | 평균 MDD -27.9%
Buy&Hold: 평균 $151.31 | 평균 MDD -29.4%

===== FORWARD =====
ticker    bot  buy_h

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
